# Homework 01A — kNN on CIFAR-10, from scratch

**Two sessions · NumPy only for the classifier · five autograded TODOs**

You will implement the full reasoning path: pixels → distances → neighbors → validation → one final test. The data and plotting code are supplied; the learning algorithm is yours. This notebook is self-contained: do not create or import a course class from another file.

> Inspired by the [Stanford CS231n kNN assignment](https://cs231n.github.io/assignments2024/assignment1/) and [image-classification notes](https://cs231n.github.io/classification/).

## Learning goals

By the end, you should be able to explain why `fit` is almost free for kNN, why prediction is expensive, why validation—not test accuracy—chooses `k`, and what raw-pixel distance gets wrong.

In [ ]:
import pickle
import tarfile
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

SEED = 42
rng = np.random.default_rng(SEED)
CLASS_NAMES = np.array([
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
])
print('NumPy', np.__version__)

## 1. Meet the data

CIFAR-10 contains 32×32 color images from ten classes. The loader below downloads the official Python archive once and caches it. We use a subset so a from-scratch distance matrix remains comfortable on a CPU.

In [ ]:
def load_cifar10(cache_dir='data'):
    cache = Path(cache_dir)
    archive = cache / 'cifar-10-python.tar.gz'
    root = cache / 'cifar-10-batches-py'
    cache.mkdir(parents=True, exist_ok=True)
    if not root.exists():
        if not archive.exists():
            print('Downloading CIFAR-10 (about 163 MB)...')
            urllib.request.urlretrieve(
                'https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz', archive
            )
        with tarfile.open(archive, 'r:gz') as tar:
            tar.extractall(cache, filter='data')

    def read_batch(path):
        with open(path, 'rb') as handle:
            batch = pickle.load(handle, encoding='bytes')
        images = batch[b'data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
        return images, np.asarray(batch[b'labels'])

    train_parts = [read_batch(root / f'data_batch_{i}') for i in range(1, 6)]
    X_train = np.concatenate([part[0] for part in train_parts])
    y_train = np.concatenate([part[1] for part in train_parts])
    X_test, y_test = read_batch(root / 'test_batch')
    return X_train, y_train, X_test, y_test

X_all, y_all, X_test_all, y_test_all = load_cifar10()
train_ids = rng.choice(len(X_all), size=5000, replace=False)
test_ids = rng.choice(len(X_test_all), size=1000, replace=False)
X_images, y = X_all[train_ids], y_all[train_ids]
X_test_images, y_test = X_test_all[test_ids], y_test_all[test_ids]

fig, axes = plt.subplots(2, 5, figsize=(11, 4.8))
for label, ax in enumerate(axes.flat):
    idx = np.flatnonzero(y == label)[0]
    ax.imshow(X_images[idx])
    ax.set_title(CLASS_NAMES[label])
    ax.axis('off')
plt.suptitle('One training example per CIFAR-10 class')
plt.tight_layout()

We flatten each image into a vector of 3,072 numbers and scale bytes to `[0, 1]`. With squared Euclidean distance, the nearest image is unchanged if we omit the square root.

In [ ]:
X = X_images.reshape(len(X_images), -1).astype(np.float32) / 255.0
X_test = X_test_images.reshape(len(X_test_images), -1).astype(np.float32) / 255.0
print('train:', X.shape, y.shape, 'test:', X_test.shape, y_test.shape)

## TODO 1 — Pairwise distances with broadcasting

Write `squared_l2_distances(X_query, X_ref)`. The result must have shape `(number of queries, number of references)` and entry `(i, j)` must be $\lVert q_i-r_j\rVert_2^2$. Use NumPy broadcasting or the expansion $\lVert q-r\rVert^2=\lVert q\rVert^2+\lVert r\rVert^2-2q^Tr$; do not loop over examples.

Helpful reading: [NumPy broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html). Before coding, write the shapes of the three terms on paper.

In [ ]:
def squared_l2_distances(X_query, X_ref):
    # TODO 1: return all pairwise squared distances without example loops.
    # Hint: clip tiny negative values caused by floating-point arithmetic to zero.
    raise NotImplementedError('TODO 1')

In [ ]:
def check_todo_1(fn):
    query = np.array([[0., 0.], [1., 2.]])
    ref = np.array([[0., 1.], [2., 2.], [-1., 0.]])
    expected = ((query[:, None, :] - ref[None, :, :]) ** 2).sum(axis=2)
    got = fn(query, ref)
    assert got.shape == (2, 3), f'Expected shape (2, 3), got {got.shape}'
    np.testing.assert_allclose(got, expected, atol=1e-7)
    assert np.all(got >= 0), 'Squared distances cannot be negative.'
    print('✓ TODO 1 passed: distances, shape, and non-negativity')

check_todo_1(squared_l2_distances)

## TODO 2 — A classifier that memorizes

Implement `fit`. A kNN model has no weight optimization: training stores the examples and labels. Validate incompatible inputs rather than silently accepting them. This unusual `fit` method is exactly why writing the class here is useful: the interface is familiar, but the algorithm's behavior is visible.

In [ ]:
class KNearestNeighbor:
    def fit(self, X_train, y_train):
        # TODO 2: validate and store the training arrays; return self.
        raise NotImplementedError('TODO 2')

    def distances(self, X_query):
        if not hasattr(self, 'X_train'):
            raise RuntimeError('Call fit before distances.')
        return squared_l2_distances(X_query, self.X_train)

    def predict_from_distances(self, dists, k=1):
        # TODO 3 goes here.
        raise NotImplementedError('TODO 3')

    def predict(self, X_query, k=1):
        return self.predict_from_distances(self.distances(X_query), k=k)

In [ ]:
def check_todo_2(cls):
    features = np.arange(12).reshape(4, 3)
    labels = np.array([2, 0, 1, 2])
    model = cls()
    assert model.fit(features, labels) is model, 'fit should return self.'
    np.testing.assert_array_equal(model.X_train, features)
    np.testing.assert_array_equal(model.y_train, labels)
    try:
        cls().fit(features, labels[:2])
    except ValueError:
        pass
    else:
        raise AssertionError('Reject different numbers of examples and labels.')
    print('✓ TODO 2 passed: fit stores valid training data')

check_todo_2(KNearestNeighbor)

## TODO 3 — Let the closest labels vote

For each query row, find the indices of the `k` smallest distances, retrieve their labels, and return the majority label. Resolve ties deterministically by choosing the smallest class label. `np.argsort`, integer indexing, and `np.bincount` are enough.

In [ ]:
def check_todo_3(cls):
    model = cls().fit(np.zeros((5, 2)), np.array([2, 1, 1, 2, 0]))
    dists = np.array([[0.4, 0.1, 0.2, 0.3, 0.5], [0.1, 0.2, 9., 8., 7.]])
    np.testing.assert_array_equal(model.predict_from_distances(dists, k=3), [1, 1])
    # With labels 2 and 1 tied, np.argmax(np.bincount(...)) selects label 1.
    np.testing.assert_array_equal(model.predict_from_distances(dists[:1], k=2), [1])
    assert model.predict_from_distances(dists, k=1).dtype.kind in 'iu'
    print('✓ TODO 3 passed: nearest labels, majority vote, and tie rule')

check_todo_3(KNearestNeighbor)

### Pause and inspect neighbors

Predict 100 held-out images with `k=1`. Before running the cell, predict whether raw pixels will retrieve objects with the same **identity**, **color**, **pose**, or **background**.

In [ ]:
demo_model = KNearestNeighbor().fit(X[:1000], y[:1000])
demo_dists = demo_model.distances(X[1000:1100])
demo_pred = demo_model.predict_from_distances(demo_dists, k=1)
nearest = np.argmin(demo_dists, axis=1)

fig, axes = plt.subplots(4, 6, figsize=(10, 7))
for row, q in enumerate(range(4)):
    axes[row, 0].imshow(X_images[1000 + q])
    axes[row, 0].set_title(f'query\n{CLASS_NAMES[y[1000 + q]]}')
    axes[row, 1].text(.5, .5, '→', ha='center', va='center', fontsize=24)
    axes[row, 2].imshow(X_images[nearest[q]])
    axes[row, 2].set_title(f'nearest\n{CLASS_NAMES[demo_pred[q]]}')
    for col in range(3, 6):
        axes[row, col].axis('off')
for ax in axes.flat:
    ax.axis('off')
plt.tight_layout()

**Reflection 1.** Describe one pair where pixel distance behaves sensibly and one where it fails. What visual property appears to dominate?

## TODO 4 — Make one honest split

Write a reproducible function returning disjoint training and validation indices. Shuffle with `np.random.default_rng(seed)`, use `val_fraction` of the examples for validation, and place every remaining example in training. Do not touch the test set.

In [ ]:
def train_val_split(n_examples, val_fraction=0.2, seed=42):
    # TODO 4: return train_idx, val_idx as one-dimensional integer arrays.
    raise NotImplementedError('TODO 4')

In [ ]:
def check_todo_4(fn):
    train_idx, val_idx = fn(20, val_fraction=0.25, seed=7)
    assert len(train_idx) == 15 and len(val_idx) == 5
    assert len(np.intersect1d(train_idx, val_idx)) == 0
    np.testing.assert_array_equal(np.sort(np.r_[train_idx, val_idx]), np.arange(20))
    train_2, val_2 = fn(20, val_fraction=0.25, seed=7)
    np.testing.assert_array_equal(train_idx, train_2)
    np.testing.assert_array_equal(val_idx, val_2)
    print('✓ TODO 4 passed: complete, disjoint, reproducible split')

check_todo_4(train_val_split)

## TODO 5 — Evaluate one choice of k

Implement a single validation experiment. Fit only on `train_idx`, predict only `val_idx`, and return accuracy as a float in `[0, 1]`. This separates the *mechanism* of evaluation from the repeated cross-validation loop supplied afterward.

In [ ]:
def validation_accuracy(X, y, train_idx, val_idx, k):
    # TODO 5: train, predict, and compute the fraction correct.
    raise NotImplementedError('TODO 5')

In [ ]:
def check_todo_5(fn):
    features = np.array([[0.], [0.2], [9.8], [10.], [0.1], [9.9]])
    labels = np.array([0, 0, 1, 1, 0, 1])
    score = fn(features, labels, np.arange(4), np.array([4, 5]), k=1)
    assert isinstance(score, (float, np.floating))
    np.testing.assert_allclose(score, 1.0)
    print('✓ TODO 5 passed: fit and validation data remain separated')

check_todo_5(validation_accuracy)

## Cross-validation — choose k without looking at the test set

Now that you have implemented one split, the experiment below repeats the logic across five folds. Each fold becomes validation once. The curve is an **inverted loss curve**: higher accuracy is better. Error bars show variation across folds; they matter when several values of `k` are nearly tied.

In [ ]:
def cross_validate_knn(X, y, k_values, n_folds=5, seed=42):
    order = np.random.default_rng(seed).permutation(len(X))
    folds = np.array_split(order, n_folds)
    scores = {k: [] for k in k_values}
    for fold_id in range(n_folds):
        val_idx = folds[fold_id]
        train_idx = np.concatenate([folds[j] for j in range(n_folds) if j != fold_id])
        model = KNearestNeighbor().fit(X[train_idx], y[train_idx])
        # Reuse this expensive matrix for every k.
        dists = model.distances(X[val_idx])
        for k in k_values:
            pred = model.predict_from_distances(dists, k=k)
            scores[k].append(float(np.mean(pred == y[val_idx])))
        print(f'finished fold {fold_id + 1}/{n_folds}')
    return scores

K_VALUES = [1, 3, 5, 7, 9, 12, 15, 20]
cv_scores = cross_validate_knn(X, y, K_VALUES)
means = np.array([np.mean(cv_scores[k]) for k in K_VALUES])
stds = np.array([np.std(cv_scores[k]) for k in K_VALUES])
best_k = K_VALUES[int(np.argmax(means))]

plt.figure(figsize=(8, 4.5))
plt.errorbar(K_VALUES, means, yerr=stds, marker='o', capsize=4)
plt.axvline(best_k, color='tomato', linestyle='--', label=f'best mean: k={best_k}')
plt.xlabel('k (number of neighbors)')
plt.ylabel('validation accuracy — higher is better')
plt.title('Five-fold cross-validation chooses model complexity')
plt.xticks(K_VALUES)
plt.grid(alpha=.2)
plt.legend()
plt.show()
print({k: f'{np.mean(v):.3f} ± {np.std(v):.3f}' for k, v in cv_scores.items()})

**Reflection 2.** Which `k` do you choose? If another value has overlapping error bars, explain why the apparent winner may not be meaningfully better.

**Reflection 3.** Why would choosing `k` from test accuracy make the final number less trustworthy?

## One final test

Only now may the test set be used. Fit on all 5,000 development examples, evaluate the selected `k` once, and report the result.

In [ ]:
final_model = KNearestNeighbor().fit(X, y)
test_pred = final_model.predict(X_test, k=best_k)
test_accuracy = np.mean(test_pred == y_test)
print(f'Final k={best_k} test accuracy: {test_accuracy:.3%}')

per_class = []
for label, name in enumerate(CLASS_NAMES):
    mask = y_test == label
    per_class.append(np.mean(test_pred[mask] == y_test[mask]))
plt.figure(figsize=(10, 4))
plt.bar(CLASS_NAMES, per_class, color='steelblue')
plt.ylabel('test accuracy')
plt.title('Which classes work with raw-pixel neighbors?')
plt.xticks(rotation=35, ha='right')
plt.ylim(0, 1)
plt.tight_layout()

## Final diagnosis

1. Identify the strongest and weakest classes. Propose a visual reason for the difference.
2. Inspect at least three test images: one correct, one incorrect, and one surprising nearest neighbor. Discuss what the distance function notices.
3. State the computational cost of storing the training data and predicting one query against $N$ training images of dimension $D$.
4. Give one reason a learned linear classifier might improve on raw-pixel kNN—and one limitation it may still retain.

## Submission checklist

- [ ] Restart kernel and run all cells from top to bottom.
- [ ] Five green TODO checks are visible.
- [ ] Cross-validation and per-class plots are visible.
- [ ] Reflections 1–3 and the final diagnosis are answered.
- [ ] The test set was used only in the final section.